In [1]:
%pip install nltk

  Using cached click-8.4.2-py3-none-any.whl.metadata (2.6 kB)
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ------------------------------------ --- 1.6/1.7 MB 11.5 MB/s eta 0:00:01
   ---------------------------------------- 1.7/1.7 MB 11.0 MB/s  0:00:00
Using cached click-8.4.2-py3-none-any.whl (119 kB)
   ---------------------------------------- 0.0/676.7 kB ? eta -:--:--
   ---------------------------------------- 676.7/676.7 kB 6.8 MB/s  0:00:00

   ---------------------------------------- 0/5 [tqdm]
   ---------------------------------------- 0/5 [tqdm]
   ---------------------------------------- 0/5 [tqdm]
   ---------------------------------------- 0/5 [tqdm]
   -------- ------------------------------- 1/5 [regex]
   -------- ------------------------------- 1/5 [regex]
   ---------------- ----------------------- 2/5 [defusedxml]
   ------------------------ --------------- 3/5 [click]
   ------------------------ --------------- 3/5 [click]
   ----------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer

from sklearn.feature_extraction.text import CountVectorizer,TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report,accuracy_score,confusion_matrix,ConfusionMatrixDisplay

import os
import string
import warnings
warnings.filterwarnings('ignore')
#Download required NLTK packages (tokenizers and stopwords lists)
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')

print('All libraries loaded and NLTK resouces downloaded successfully!')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\mrnam\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\mrnam\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\mrnam\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.


All libraries loaded and NLTK resouces downloaded successfully!


In [9]:
# Text Processing 
sample_text="Hello World! We are learning text processing today!"
print("Original Text:",sample_text)

# 1. Lowercasing
text_lower=sample_text.lower()
print("1. Lowercasing Output:",text_lower)

# 2. Tokenization
tokens=word_tokenize(text_lower)
print("2. Tokenization Output (Tokens):",tokens)

# 3. Removing Punchuation and special characters
tokens_no_punct=[word for word in tokens if word not in string.punctuation]
print("3. Punctuation Removed:",tokens_no_punct)

# 4. Removing stop words
stop_words=set(stopwords.words('english'))#set for unique words
tokens_no_stopwords=[word for word in tokens_no_punct if word not in stop_words]
print("4. Stopwords Removed:",tokens_no_stopwords)

# 5. Stemming (using PorterStemmer):- reducing words to base word(like learning or learns or learned to learn)
stemmer=PorterStemmer()
stemmed_tokens=[stemmer.stem(word) for word in tokens_no_stopwords]
print("5. Stemmed Tokens:",stemmed_tokens)

# Join back into a singe clean string
clean_sentence=" ".join(stemmed_tokens)
print("Final Preprocessed Text:",clean_sentence)


Original Text: Hello World! We are learning text processing today!
1. Lowercasing Output: hello world! we are learning text processing today!
2. Tokenization Output (Tokens): ['hello', 'world', '!', 'we', 'are', 'learning', 'text', 'processing', 'today', '!']
3. Punctuation Removed: ['hello', 'world', 'we', 'are', 'learning', 'text', 'processing', 'today']
4. Stopwords Removed: ['hello', 'world', 'learning', 'text', 'processing', 'today']
5. Stemmed Tokens: ['hello', 'world', 'learn', 'text', 'process', 'today']
Final Preprocessed Text: hello world learn text process today


In [19]:
# Text Vectorization (Turning Words into Numbers)
toy_corpus=[
    "The cat chased the mouse.",
    "The dog chased the cat.",
    "The bird sang in the tree."
]
print("===Bag of Words Representation (Counts)===")
count_vect=CountVectorizer()
bow_matrix=count_vect.fit_transform(toy_corpus)

#Display as DataFrame for visual readability
df_bow=pd.DataFrame(bow_matrix.toarray(),columns=count_vect.get_feature_names_out())
display(df_bow)

print("\n===TF-IDF Representation (Scaled Weights)===")
tfidf_vect=TfidfVectorizer()
tfidf_matrix=tfidf_vect.fit_transform(toy_corpus)

#Display as DataFrame for visual readability
df_tfidf=pd.DataFrame(tfidf_matrix.toarray(),columns=tfidf_vect.get_feature_names_out())
display(df_tfidf.round(3))

===Bag of Words Representation (Counts)===


,bird,cat,chased,dog,in,mouse,sang,the,tree
0,0,1,1,0,0,1,0,2,0
1,0,1,1,1,0,0,0,2,0
2,1,0,0,0,1,0,1,2,1



===TF-IDF Representation (Scaled Weights)===


,bird,cat,chased,dog,in,mouse,sang,the,tree
0,0.000,0.404,0.404,0.000,0.000,0.531,0.000,0.627,0.000
1,0.000,0.404,0.404,0.531,0.000,0.000,0.000,0.627,0.000
2,0.431,0.000,0.000,0.000,0.431,0.000,0.431,0.509,0.431


In [25]:
# Load Dataset (by default read in utf-8 but it cann't read text or language so using latin 1)
df=pd.read_csv('spam.csv',encoding='latin-1')
#Select only label and text and rename them
df=df[['v1','v2']].rename(columns={'v1':'label','v2':'text'})
df.head()
#Binary encode label (0=not spam,1=spam)
df['is_spam']=df['label'].map({'ham':0,'spam':1})
print(f"Dataset Shape: {df.shape[0]} rows and {df.shape[1]} columns.\n")
print("Class Distribution (ham vs spam)")
print(df['label'].value_counts())
print("First 5 rows:")
display(df.head())

Dataset Shape: 5572 rows and 3 columns.

Class Distribution (ham vs spam)
label
ham     4825
spam     747
Name: count, dtype: int64
First 5 rows:


,label,text,is_spam
0,ham,"Go until jurong point, crazy.. Available only ...",0
1,ham,Ok lar... Joking wif u oni...,0
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,1
3,ham,U dun say so early hor... U c already then say...,0
4,ham,"Nah I don't think he goes to usf, he lives aro...",0


In [29]:
# Applying cleaning pipeline
def preprocess_text(text):
    # 1. Lowercase
    text=text.lower()
    # 2. Tokenizer
    tokens=word_tokenize(text)
    # 3. Remove Punctuation
    tokens=[word for word in tokens if word not in string.punctuation]
    # 4. Remove stopwords
    stop_words=set(stopwords.words('english'))
    tokens=[word for word in tokens if word not in stop_words]
    # 5. Stemming
    tokens=[stemmer.stem(word) for word in tokens]
    #Join back into string
    return " ".join(tokens)

print("Cleaning all SMS messages in the dataset... (this may take a few seconds)")
df['clean_text']=df['text'].apply(preprocess_text)
print("Preprocessing complete!")

#Compare a sample
print("Original:",df['text'].iloc[1])
print("Cleaned:",df['clean_text'].iloc[1])

Cleaning all SMS messages in the dataset... (this may take a few seconds)
Preprocessing complete!
Original: Ok lar... Joking wif u oni...
Cleaned: ok lar ... joke wif u oni ...


In [ ]:
# Data Splitting
x_train,x_test,y_train,y_test=train_test_split(df['clean_text'],df['is_spam'],test_size=0.20,random_state=42,stratify=df['is_spam'])
print(f"Train size: {x_train.shape[0]} | Test size: {x_test.shape[0]}")
# 2. Vectorize (limiting features to the top 2500 most important words)
tfidf=TfidfVectorizer(max_features=2500)
x_train_tfidf=tfidf.fit_transform(x_train)
x_test_tfidf=tfidf.transform(x_test)